# 04. Warehouse Segmentation & K-Means Clustering
## Supply Chain Optimization — FMCG / Retail

---

### Objective
Segment distribution centers based on multi-year operational behavior:
1. Evaluate K=2..8 using **Elbow Method (Inertia)** and **Silhouette Scores**.
2. Determine mathematically optimal number of clusters.
3. Fit K-Means on normalized operational feature profiles.
4. Profile operational segments (e.g., High-Velocity Shortage Risk, Balanced Flow, Chronic Overstock).


In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.clustering import WarehouseClusterer

df_profiles = pd.read_csv("../data/processed/warehouse_profiles.csv")
clusterer = WarehouseClusterer(k_range=(2, 8), random_state=42)


### 1. Evaluate K via Elbow and Silhouette Scores


In [ ]:
k_eval_df = clusterer.evaluate_k_range(df_profiles)
display(k_eval_df)


In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4))

color = "tab:blue"
ax1.set_xlabel("Number of Clusters (K)", fontsize=11)
ax1.set_ylabel("Inertia (Elbow Method)", color=color, fontsize=11)
ax1.plot(k_eval_df["k"], k_eval_df["inertia"], marker="o", color=color, linewidth=2)
ax1.tick_params(axis="y", labelcolor=color)

ax2 = ax1.twinx()
color = "tab:red"
ax2.set_ylabel("Silhouette Score", color=color, fontsize=11)
ax2.plot(k_eval_df["k"], k_eval_df["silhouette_score"], marker="s", color=color, linestyle="--", linewidth=2)
ax2.tick_params(axis="y", labelcolor=color)

plt.title("Elbow Method & Silhouette Score vs. Number of Clusters K", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


### 2. Fit K-Means (K=3) & Dimensionality Reduction Mapping


In [ ]:
df_clustered = clusterer.fit_predict(df_profiles, k=3)
df_clustered.to_csv("../data/processed/warehouse_profiles.csv", index=False)

plt.figure(figsize=(9, 5))
sns.scatterplot(
    data=df_clustered,
    x="pca_x",
    y="pca_y",
    hue="cluster_name",
    style="cluster_name",
    s=100,
    palette="tab10"
)
plt.title("2D PCA Projection of Warehouse Operational Clusters", fontsize=13, fontweight="bold")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


### 3. Discovered Cluster Profiles & Operational Characteristics


In [ ]:
summary_table = clusterer.get_cluster_summary()
display(summary_table[["Operational_Segment", "demand_to_supply_ratio_avg", "capacity_utilization_avg", "inventory_turnover_avg", "stockout_incidents_sum"]])
